In [81]:
import torch
# print(torch.version.cuda)
torch.cuda.is_available()
# print(torch.backends.cudnn.enabled)

True

In [1]:
import snntorch as snn
from snntorch import surrogate
from snntorch import backprop
from snntorch import functional as SF
from snntorch import utils
from snntorch import spikeplot as splt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn.functional as F

import matplotlib.pyplot as plt
import numpy as np
import itertools

C:\Users\SHAURYAM DUBEY\AppData\Local\Temp\ipykernel_28824\1406316218.py:3: DeprecationWarning: The module snntorch.backprop will be deprecated in  a future release. Writing out your own training loop will lead to substantially faster performance.
  from snntorch import backprop


In [2]:
# Define hyperparameters
batch_size = 64

# Load FER2013 dataset
transform = transforms.Compose([
    transforms.Grayscale(),
    transforms.Resize((48, 48)),
    transforms.RandomHorizontalFlip(),
    # transforms.RandomRotation(10),
    # transforms.ColorJitter(brightness=0.2, contrast=0.2),  # Change brightness and contrast
    # transforms.RandomAffine(degrees=10, translate=(0.1, 0.1)),  # Add small shifts
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

# Load the dataset
train_dataset = datasets.ImageFolder(
    root='./dataset/train', transform=transform)
test_dataset = datasets.ImageFolder(
    root='./dataset/test', transform=transform)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [5]:
class CSNN(nn.Module):
    def __init__(self,beta=0.9, slope=25):
        super(CSNN, self).__init__()

        # Convolutional layers with 12 filters in first layer, 64 filters in second, etc.
        self.conv1 = nn.Conv2d(1, 32, 5)  # Input is 1 channel (grayscale)
        self.lif1 = snn.Leaky(
            beta=beta, spike_grad=surrogate.fast_sigmoid(slope=slope))
        # self.dropout1 = nn.Dropout(p=0.1)

        self.conv2 = nn.Conv2d(32, 64, 5)
        self.lif2 = snn.Leaky(
            beta=beta, spike_grad=surrogate.fast_sigmoid(slope=slope))
        # self.dropout2 = nn.Dropout(p=0.1)

        # Adjusted for smaller output size after convolution and pooling
        self.fc1 = nn.Linear(64 * 9 * 9, 128)
        self.lif3 = snn.Leaky(
            beta=beta, spike_grad=surrogate.fast_sigmoid(slope=slope))
        # self.dropout3 = nn.Dropout(p=0.1)

        self.fc2 = nn.Linear(128, 7)  # 7 classes for FER2013 emotions
        self.lif4 = snn.Leaky(
            beta=beta, spike_grad=surrogate.fast_sigmoid(slope=slope))

    def forward(self, x):
        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()
        mem4 = self.lif4.init_leaky()

        # Forward pass through convolution and pooling layers
        cur1 = F.max_pool2d(self.conv1(x), 2)
        spk1, mem1 = self.lif1(cur1, mem1)
        # spk1 = self.dropout1(spk1)

        cur2 = F.max_pool2d(self.conv2(spk1), 2)
        spk2, mem2 = self.lif2(cur2, mem2)
        # spk2 = self.dropout2(spk2)

        cur3 = self.fc1(spk2.view(spk2.size(0), -1))
        spk3, mem3 = self.lif3(cur3, mem3)
        # spk3 = self.dropout3(spk3)

        out = self.fc2(spk3)
        spk4, mem4 = self.lif4(out, mem4)

        return out, spk4, mem4

In [6]:
net = CSNN(beta=0.9).to(device)

In [ ]:
#original beta = 0.9 lr = 0.0001 epochs = 100 accuracy = 49.33% loss < 0.9
#2nd try - beta = 0.99 lr = 0.0001 epochs = 20 accuracy = 35.64% loss 1.6463
# 3nd try - beta = 0.95 lr = 0.0001 epochs = 20 loss 1.6640
# 5nd try - beta = 0.9 lr = 0.001 epochs = 20 accuracy = 28.66% loss 1.7209
# 6nd try - beta = 0.9 lr = 0.0001 epochs = 20 accuracy = 35.62% loss 1.6436
# 7nd try - beta = 0.8 lr = 0.0001 epochs = 20 accuracy = 33.92% loss 1.6602
# 8nd try - beta = 0.8 lr = 0.001 epochs = 20 loss 1.7313
# 9nd try - beta = 0.7 lr = 0.0001 epochs = 20 loss 1.6574
# 10nd try - beta = 0.5 lr = 0.0001 epochs = 20  loss 1.6670
# 11nd try - beta = 0.65 lr = 0.0001 epochs = 20 loss 1.6332
# 8nd try - beta = 0.7 lr = 0.0001 epochs = 20 accuracy = 33.92% loss 1.6574

In [ ]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.0001)
num_epochs = 150
num_steps = 100
# Training loop


def train_snn(num_epochs):
    for epoch in range(num_epochs):
        net.train()
        running_loss = 0
        for images, labels in train_loader:
            # images = add_noise(images)
            images, labels = images.to(device), labels.to(device)
            labels = labels.long()

            # Forward pass
            optimizer.zero_grad()
            spk_rec, a, _ = net(images)
            # spk_rec= forward_pass(net, num_steps, data)
            # print(spk_rec.size(),epoch)
            # spk_rec.squeeze(1)
            # labels = labels.view(-1)
            # print(spk_rec.size())
            loss = loss_fn(spk_rec, labels)

            # Backward pass and optimization
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {running_loss/len(train_loader):.4f}')


train_snn(num_epochs)

Epoch [1/100], Loss: 1.6566
Epoch [2/100], Loss: 1.6517
Epoch [3/100], Loss: 1.6454
Epoch [4/100], Loss: 1.6384
Epoch [5/100], Loss: 1.6395
Epoch [6/100], Loss: 1.6391
Epoch [7/100], Loss: 1.6316
Epoch [8/100], Loss: 1.6241
Epoch [9/100], Loss: 1.6198
Epoch [10/100], Loss: 1.6172
Epoch [11/100], Loss: 1.6134
Epoch [12/100], Loss: 1.6146
Epoch [13/100], Loss: 1.6066
Epoch [14/100], Loss: 1.6027
Epoch [15/100], Loss: 1.6018
Epoch [16/100], Loss: 1.6018
Epoch [17/100], Loss: 1.5973
Epoch [18/100], Loss: 1.5943
Epoch [19/100], Loss: 1.5917
Epoch [20/100], Loss: 1.5894
Epoch [21/100], Loss: 1.5841
Epoch [22/100], Loss: 1.5809
Epoch [23/100], Loss: 1.5781
Epoch [24/100], Loss: 1.5691
Epoch [25/100], Loss: 1.5729
Epoch [26/100], Loss: 1.5675
Epoch [27/100], Loss: 1.5649
Epoch [28/100], Loss: 1.5617
Epoch [29/100], Loss: 1.5593
Epoch [30/100], Loss: 1.5546
Epoch [31/100], Loss: 1.5526
Epoch [32/100], Loss: 1.5492
Epoch [33/100], Loss: 1.5518
Epoch [34/100], Loss: 1.5464
Epoch [35/100], Loss: 1

In [80]:
def evaluate():
    correct = 0
    total = 0
    net.eval()

    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs, a, _ = net(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    print(f'Test Accuracy: {100 * correct / total:.2f}%')


evaluate()

Test Accuracy: 45.26%


In [ ]:
# torch.save(net.state_dict(), 'csnn.pth')

In [ ]:
num_steps = 64
loss_fn = nn.CrossEntropyLoss()
def forward_pass(net, num_steps, data):
  mem_rec = []
  spk_rec = []
  out_rec = []
  utils.reset(net)  # resets hidden states for all LIF neurons in net

  for step in range(num_steps):
      out, spk_out, mem_out = net(data)
      spk_rec.append(spk_out)
      mem_rec.append(mem_out)
      out_rec.append(out)

  return torch.stack(out_rec), torch.stack(spk_rec), torch.stack(mem_rec)

def batch_accuracy(train_loader, net, num_steps):
  with torch.no_grad():
    total = 0
    acc = 0
    net.eval()

    train_loader = iter(train_loader)
    for data, targets in train_loader:
      data = data.to(device)
      targets = targets.to(device)
      out, spk_rec, _ = forward_pass(net, num_steps, data)
      print(out.view(out.size(0),-1))
      acc += loss_fn(out, targets) * out.size(1)
      total += out.size(1)
      out = out.mean(dim=0)  # Average across the time dimension

            # Ensure output shape is [batch_size, num_classes]
            out = out.view(out.size(0), -1)

            # Calculate loss (CrossEntropyLoss requires specific shape)
            acc += loss_fn(out, targets) * out.size(0)
            total += out.size(0)

  return acc/total

In [ ]:
optimizer = torch.optim.Adam(net.parameters(), lr=0.0001)
num_epochs = 10
loss_hist = []
test_acc_hist = []
counter = 0
num_steps = 64
# Outer training loop
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}\n-------------------------------")
    # Training loop
    for data, targets in iter(train_loader):
        data = data.to(device)
        targets = targets.to(device)

        # forward pass
        net.train()
        out, spk_rec, _ = forward_pass(net, num_steps, data)
        
        # out = out.mean(dim=1)

        # initialize the loss & sum over time
        loss_val = loss_fn(out, targets)

        # Gradient calculation + weight update
        optimizer.zero_grad()
        loss_val.backward()
        optimizer.step()

        # Store loss history for future plotting
        loss_hist.append(loss_val.item())

        if counter % 50 == 0:
          with torch.no_grad():
            net.eval()

            # Test set forward pass
            test_acc = batch_accuracy(test_loader, net, num_steps)
            print(f"Iteration {counter}, Test Acc: {test_acc * 100:.2f}%\n")
            test_acc_hist.append(test_acc.item())

        counter += 1

Epoch 1
-------------------------------
tensor([[ 0.0028,  0.0410, -0.0776,  ...,  0.0460,  0.0308, -0.0497],
        [ 0.0028,  0.0410, -0.0776,  ...,  0.0460,  0.0308, -0.0497],
        [ 0.0028,  0.0410, -0.0776,  ...,  0.0460,  0.0308, -0.0497],
        ...,
        [ 0.0028,  0.0410, -0.0776,  ...,  0.0460,  0.0308, -0.0497],
        [ 0.0028,  0.0410, -0.0776,  ...,  0.0460,  0.0308, -0.0497],
        [ 0.0028,  0.0410, -0.0776,  ...,  0.0460,  0.0308, -0.0497]],
       device='cuda:0')


RuntimeError: Expected target size [64, 7], got [64]